# 06 — Canonical panel: RTT demand + KH03 capacity

Left-join the KH03 provider-month capacity onto the flagged RTT panel. KH03 has no specialty dimension (sectors only), so a provider's capacity repeats across its specialty rows — a documented limitation, not a bug. Independents are not in KH03 (it covers NHS trusts), so their capacity columns stay empty.

In [1]:
import pandas as pd
from pathlib import Path

PROC = Path("../data/processed")
rtt = pd.read_csv(PROC / "rtt_panel_with_provider_flags.csv",
                  parse_dates=["month", "first_month", "last_month"])
kh = pd.read_csv(PROC / "kh03_monthly_panel.csv", parse_dates=["month"])

assert not kh.duplicated(["provider_code", "month"]).any()   # a duplicate here would multiply rtt rows
panel = rtt.merge(kh, on=["provider_code", "month"], how="left")
assert len(panel) == len(rtt), (len(panel), len(rtt))        # left join must preserve the row count
print(panel.shape)

(343441, 24)


## Match rates

Denominator: rows in each group. Expect high coverage for public trusts and zero for independents (KH03 is a trust return).

In [2]:
has_cap = panel["avail_ga"].notna()
print("overall: %.1f%% of rows carry capacity" % (100 * has_cap.mean()))
print(panel.groupby("is_public")["avail_ga"].apply(lambda s: s.notna().mean()).rename("match_rate"))

# which public rows are missing capacity - months before/after a trust existed, or non-returners
miss = panel[panel["is_public"] & ~has_cap]
print(miss["provider_code"].nunique(), "public providers have some unmatched months,",
      len(miss), "rows of", int(panel["is_public"].sum()), "public rows")

overall: 66.2% of rows carry capacity
is_public
False    0.001054
True     0.990388
Name: match_rate, dtype: float64
13 public providers have some unmatched months, 2207 rows of 229600 public rows


In [3]:
# spot check: one large acute trust, month by month
one = panel[(panel["provider_code"] == "R0A") & (panel["tfunc_code"] == "C_110")].sort_values("month")
one[["month", "total_waiting", "pct_gt18", "avail_ga", "occupancy_ga"]].head(12)

,month,total_waiting,pct_gt18,avail_ga,occupancy_ga
286280,2018-04-01,4263,0.113066,1969.186813,0.898446
324129,2018-05-01,4185,0.102987,1969.186813,0.898446
313773,2018-06-01,4104,0.124513,1969.186813,0.898446
303539,2018-07-01,4378,0.113522,1960.945652,0.881457
276154,2018-08-01,4502,0.121279,1960.945652,0.881457
334474,2018-09-01,4451,0.126938,1960.945652,0.881457
331041,2018-10-01,4584,0.121291,2012.293478,0.888328
327603,2018-11-01,4631,0.118549,2012.293478,0.888328
289762,2018-12-01,4573,0.142139,2012.293478,0.888328
296668,2019-01-01,4384,0.151004,2018.755556,0.908866


In [4]:
panel.to_csv(PROC / "rtt_kh03_canonical_panel.csv", index=False)
print(panel.shape, "->", PROC / "rtt_kh03_canonical_panel.csv")
print(list(panel.columns))

(343441, 24) -> ..\data\processed\rtt_kh03_canonical_panel.csv
['month', 'provider_code', 'tfunc_code', 'total_waiting', 'gt18', 'gt52', 'pct_gt18', 'pct_gt52', 'trust_name', 'status', 'icb_code', 'postcode', 'is_public', 'n_months_active', 'first_month', 'last_month', 'fl_client', 'fl_allocatable', 'avail_ga', 'occ_ga', 'occupancy_ga', 'avail_total', 'occ_total', 'occupancy_total']
